<a href="https://colab.research.google.com/github/Eunchae-L/ADD2023-SSAD-PASE/blob/main/ADD2023_Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
%cd /content/drive/MyDrive/STUDY/Project/ADD2023

/content/drive/MyDrive/STUDY/Project/ADD2023


In [20]:
import os, glob, math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import pandas as pd
import matplotlib.pyplot as plt

In [21]:
label_R1 = "data/test/label_R1_00.txt"
label_R2 = "data/test/label_R2_00.txt"

label_R1_df = pd.read_csv(label_R1, sep=r"[\s,]+", engine="python", header=None)
label_R2_df = pd.read_csv(label_R2, sep=r"[\s,]+", engine="python", header=None)

label_R1_df = label_R1_df.iloc[:, :2].copy()
label_R1_df.columns = ["path", "label"]

label_R2_df = label_R2_df.iloc[:, :2].copy()
label_R2_df.columns = ["path", "label"]

label_map = {
    "genuine": 0,
    "fake": 1
}

label_R1_df["label"] = label_R1_df["label"].map(label_map)
label_R2_df["label"] = label_R2_df["label"].map(label_map)

if label_R1_df["label"].isna().any():
    raise ValueError("label_R1_df에 genuine/fake 이외의 값이 있음")

if label_R2_df["label"].isna().any():
    raise ValueError("label_R2_df에 genuine/fake 이외의 값이 있음")

In [22]:
class RandomCropTestDatasetNoNorm(Dataset):
    def __init__(self, root_dir, label_df, crop_len=32000):
        self.root_dir = root_dir
        self.crop_len = crop_len

        # label_df 그대로 dict 만들기
        # path column 값이 확장자 없는 경우도 허용하기 위해 둘 다 매핑
        df = label_df.copy()

        # stringify
        df["path"] = df["path"].astype(str)

        label_by_key = {}
        for p, y in zip(df["path"].tolist(), df["label"].tolist()):
            label_by_key[p] = int(y)

        # root_dir의 파일들과 매칭
        items = []
        npy_paths = sorted(glob.glob(os.path.join(root_dir, "*.npy")))
        npy_names = {os.path.basename(p): p for p in npy_paths}
        npy_stems = {os.path.splitext(os.path.basename(p))[0]: p for p in npy_paths}

        missing = []
        for key, y in label_by_key.items():
            if key in npy_names:
                items.append((npy_names[key], key, y))
            else:
                stem = os.path.splitext(key)[0]
                if stem in npy_stems:
                    # label_df가 확장자 없이 준 케이스 대응
                    items.append((npy_stems[stem], key, y))
                else:
                    missing.append(key)

        self.items = items

    def __len__(self):
        return len(self.items)

    def _crop(self, wav: np.ndarray):
        wav = np.asarray(wav, dtype=np.float32).squeeze()
        T = wav.shape[0]
        if T >= self.crop_len:
            start = random.randint(0, T - self.crop_len)
            return wav[start:start+self.crop_len]
        else:
            seg = np.zeros((self.crop_len,), dtype=np.float32)
            seg[:T] = wav
            return seg

    def __getitem__(self, idx):
        npy_path, key, y = self.items[idx]
        wav = np.load(npy_path).astype(np.float32)
        seg = self._crop(wav)
        return torch.from_numpy(seg).float(), int(y), key


class MultiCropView(Dataset):
    def __init__(self, base_ds, num_crops: int):
        assert num_crops >= 1
        self.base = base_ds
        self.num_crops = num_crops

    def __len__(self):
        return len(self.base) * self.num_crops

    def __getitem__(self, i):
        utt_idx = i // self.num_crops
        return self.base[utt_idx]


def make_loader(ds, batch_size=32, num_workers=2, shuffle=False, pin_memory=True):
    return DataLoader(
        ds, batch_size=batch_size, shuffle=shuffle,
        num_workers=num_workers, pin_memory=pin_memory,
        persistent_workers=(num_workers > 0),
        drop_last=False
    )

In [23]:
CHANNEL_LIST = (16,32,64,128,128,256,256,512)
KERNEL_LIST = (10,8,8,4,4,4,4,4)
STRIDE_LIST = (5,4,2,2,2,2,2,2)

embedding_dim = 512
worker_input_dim = 256
proj_hidden = 512

TCN_LAYERS = 4
# DROPOUT = 0.0

In [24]:
class ConvBlock(nn.Module):
  def __init__(
      self,
      in_ch,
      out_ch,
      kernel_size,
      stride,
      # dropout,
  ):
      super().__init__()
      padding = kernel_size // 2

      self.conv = nn.Conv1d(
          in_ch,
          out_ch,
          kernel_size=kernel_size,
          stride=stride,
          padding=padding
      )

      self.bn = nn.BatchNorm1d(out_ch)
      self.act = nn.ReLU()
      # #dropout 값이 0보다 크면 Dropout 레이어를 쓰고,0이면 아무 일도 하지 않는 Identity 레이어를 쓴다.
      # self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

  def forward(self, x):
      x = self.conv(x)
      x = self.bn(x)
      x = self.act(x)
      # x = self.dropout(x)
      return x

class TCNBlock(nn.Module):
  def __init__(
      self,
      channels,
      dilation,
      kernel_size,
      causal=False
  ):
      super().__init__()
      self.causal = causal

      if causal:
          padding = (kernel_size - 1) * dilation
      else:
          padding = ((kernel_size - 1) * dilation) // 2

      self.conv = nn.Conv1d(
          channels,
          channels,
          kernel_size=kernel_size,
          dilation=dilation,
          padding=padding
      )
      self.bn = nn.BatchNorm1d(channels)
      self.act = nn.ReLU()

  def forward(self, x):
      out = self.conv(x)

      if self.causal:
          out = out[..., : x.size(-1)] #causal crop
      out = self.bn(out)
      out = self.act(out)
      return x + out

In [25]:
class Encoder(nn.Module):
  def __init__(
      self,
      channel_list,
      kernel_list,
      stride_list,
      embedding_dim=512,
      tcn_layers=4,
      tcn_kernel_size=3,
      tcn_causal=False,
      proj_hidden=512,
      worker_input_dim=256,
      # dropout=DROPOUT
  ):
      super().__init__()

      assert len(channel_list) == 8
      assert len(kernel_list) == 8
      assert len(stride_list) == 8

      self.embedding_dim = embedding_dim
      self.output_dim = worker_input_dim

      # 8 Conv1D Block
      convs = []
      in_ch = 1
      for out_ch, kernel_size, stride in zip(channel_list, kernel_list, stride_list):
          convs.append(
              ConvBlock(
                  in_ch,
                  out_ch,
                  kernel_size=kernel_size,
                  stride=stride,
                  # dropout=dropout
              )
          )
          in_ch = out_ch
      self.conv_blocks = nn.ModuleList(convs) #nn.ModuleList: 모듈들을 리스트 형태로 관리, 동적으로 모듈 추가/삭제 가능

      # skip projection
      self.skip_proj = nn.ModuleList([
          nn.Conv1d(ch, embedding_dim, kernel_size=1)
          for ch in channel_list
      ])

      # TCN
      self.tcn = nn.Sequential(*[
          TCNBlock(
              channels=embedding_dim,
              dilation=2 ** i,
              kernel_size=tcn_kernel_size,
              causal=tcn_causal
          )
          for i in range(tcn_layers)
      ])

      # Nonlinear projection head
      self.proj_head = nn.Sequential(
          nn.Conv1d(embedding_dim, proj_hidden, kernel_size=1),
          nn.Tanh(),
          nn.Conv1d(proj_hidden, self.output_dim, kernel_size=1)
      )

      self.bn = nn.BatchNorm1d(self.output_dim)

  def forward(self, x):
      """
      x: (B, T) or (B, 1, T)
      return: (B, 512, T)
      """
      if x.dim() == 2:
        x = x.unsqueeze(1)

      skip_feats = []
      out = x

      for block, proj in zip(self.conv_blocks, self.skip_proj):
        out = block(out)
        skip_feats.append(proj(out))

      #temporal alignment (crop to shortest)
      min_t = min(f.size(-1) for f in skip_feats)
      skip_feats = [f[..., :min_t] for f in skip_feats]

      z = torch.stack(skip_feats, dim=0).sum(dim=0)

      z = self.tcn(z)
      z = self.proj_head(z)
      z = self.bn(z)
      return z

In [26]:
# -------------------------
# z -> image adapter (C=256 => 16x16)
# -------------------------
class ZImageAdapter(nn.Module):
    def __init__(self, image_hw=(16, 16), pool="mean"):
        super().__init__()
        self.h, self.w = image_hw
        self.pool = pool

    def forward(self, z):
        # z: (B,C,T)
        if self.pool == "mean":
            z = z.mean(dim=-1)
        elif self.pool == "max":
            z = z.max(dim=-1).values
        else:
            raise ValueError(self.pool)

        B, C = z.shape
        if C != self.h * self.w:
            raise ValueError(f"z dim {C} != {self.h*self.w}. Set image_hw accordingly.")
        return z.view(B, 1, self.h, self.w)

In [27]:
class MFM(nn.Module):
    """
    Max-Feature-Map:
    input channels must be even. output channels = input/2
    y = max(x[:C/2], x[C/2:])
    """
    def forward(self, x):
        c = x.size(1)
        assert c % 2 == 0, f"MFM expects even channels, got {c}"
        a, b = torch.split(x, c // 2, dim=1)
        return torch.max(a, b)

def kaiming_init(m: nn.Module):
    if isinstance(m, (nn.Conv2d, nn.Linear)) and not isinstance(m, nn.LazyLinear):
        nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
        if m.bias is not None:
            nn.init.zeros_(m.bias)

class SEBlock(nn.Module):
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        hidden = max(channels // reduction, 4)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Conv2d(channels, hidden, kernel_size=1, bias=True)
        self.fc2 = nn.Conv2d(hidden, channels, kernel_size=1, bias=True)

    def forward(self, x):
        s = self.pool(x)
        s = F.relu(self.fc1(s), inplace=True)
        s = torch.sigmoid(self.fc2(s))
        return x * s

In [28]:
class LCNNBig(nn.Module):
    """
    LCNN-big (MFM 기반).
    - 입력: (B, 1, H, W)
    - FC: 160 -> MFM -> 80 -> BN -> Dropout(0.75) -> FC(2)
    """
    def __init__(self, num_classes=2, dropout=0.75):
        super().__init__()
        self.mfm = MFM()

        # Table 1 layer flow (Conv -> MFM -> Pool -> ...)
        self.conv1 = nn.Conv2d(1, 64, kernel_size=5, stride=1, padding=2)       # -> MFM -> 32
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv4 = nn.Conv2d(32, 64, kernel_size=1, stride=1, padding=0)      # -> MFM -> 32
        self.bn6   = nn.BatchNorm2d(32)
        self.conv7 = nn.Conv2d(32, 96, kernel_size=3, stride=1, padding=1)      # -> MFM -> 48
        self.pool9 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.bn10  = nn.BatchNorm2d(48)

        self.conv11 = nn.Conv2d(48, 96, kernel_size=1, stride=1, padding=0)     # -> MFM -> 48
        self.bn13   = nn.BatchNorm2d(48)
        self.conv14 = nn.Conv2d(48, 128, kernel_size=3, stride=1, padding=1)    # -> MFM -> 64
        self.pool16 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv17 = nn.Conv2d(64, 128, kernel_size=1, stride=1, padding=0)    # -> MFM -> 64
        self.bn19   = nn.BatchNorm2d(64)
        self.conv20 = nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1)     # -> MFM -> 32
        self.bn22   = nn.BatchNorm2d(32)
        self.conv23 = nn.Conv2d(32, 64, kernel_size=1, stride=1, padding=0)     # -> MFM -> 32
        self.bn25   = nn.BatchNorm2d(32)
        self.conv26 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)     # -> MFM -> 32
        self.pool28 = nn.MaxPool2d(kernel_size=2, stride=2)

        # FC part: use LazyLinear so input spatial size can vary
        self.fc29   = nn.LazyLinear(160)
        self.bn31   = nn.BatchNorm1d(80)
        self.drop   = nn.Dropout(p=dropout)
        self.fc32   = nn.Linear(80, num_classes)

        self.apply(kaiming_init)

    def extract_feat(self, x):
        # x: (B,1,H,W)
        x = self.mfm(self.conv1(x))       # 64 -> 32
        x = self.pool3(x)

        x = self.mfm(self.conv4(x))       # 64 -> 32
        x = self.bn6(x)
        x = self.mfm(self.conv7(x))       # 96 -> 48
        x = self.pool9(x)
        x = self.bn10(x)

        x = self.mfm(self.conv11(x))      # 96 -> 48
        x = self.bn13(x)
        x = self.mfm(self.conv14(x))      # 128 -> 64
        x = self.pool16(x)

        x = self.mfm(self.conv17(x))      # 128 -> 64
        x = self.bn19(x)
        x = self.mfm(self.conv20(x))      # 64 -> 32
        x = self.bn22(x)
        x = self.mfm(self.conv23(x))      # 64 -> 32
        x = self.bn25(x)
        x = self.mfm(self.conv26(x))      # 64 -> 32
        x = self.pool28(x)

        x = torch.flatten(x, 1)           # (B, *)
        x = self.fc29(x)                  # (B,160)
        # MFM on vector: split channels 160->80
        a, b = torch.split(x, 80, dim=1)
        x = torch.max(a, b)           # (B,80)

        x = self.bn31(x)
        x = self.drop(x)
        return x # embedding (B,80)

    def forward(self, x):
        return self.fc32(self.extract_feat(x))

In [29]:
class LCNNSmall(nn.Module):
    """
    LCNN-small (Table 2 기반).
    - FC: 64 -> MFM -> 32 -> BN -> Dropout(0.75) -> FC(2)
    """
    def __init__(self, num_classes=2, dropout=0.75):
        super().__init__()
        self.mfm = MFM()

        self.conv1 = nn.Conv2d(1, 16, kernel_size=5, stride=1, padding=2)       # -> MFM -> 8
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv4 = nn.Conv2d(8, 16, kernel_size=1, stride=1, padding=0)       # -> MFM -> 8
        self.bn6   = nn.BatchNorm2d(8)
        self.conv7 = nn.Conv2d(8, 24, kernel_size=3, stride=1, padding=1)       # -> MFM -> 12
        self.pool9 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.bn10  = nn.BatchNorm2d(12)

        self.conv11 = nn.Conv2d(12, 24, kernel_size=1, stride=1, padding=0)     # -> MFM -> 12
        self.bn13   = nn.BatchNorm2d(12)
        self.conv14 = nn.Conv2d(12, 24, kernel_size=3, stride=1, padding=1)     # -> MFM -> 12
        self.pool16 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv17 = nn.Conv2d(12, 24, kernel_size=1, stride=1, padding=0)     # -> MFM -> 12
        self.bn19   = nn.BatchNorm2d(12)
        self.conv20 = nn.Conv2d(12, 8, kernel_size=3, stride=1, padding=1)      # -> MFM -> 4
        self.bn22   = nn.BatchNorm2d(4)
        self.conv23 = nn.Conv2d(4, 8, kernel_size=1, stride=1, padding=0)       # -> MFM -> 4
        self.bn25   = nn.BatchNorm2d(4)
        self.conv26 = nn.Conv2d(4, 8, kernel_size=3, stride=1, padding=1)       # -> MFM -> 4
        self.pool28 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.fc29   = nn.LazyLinear(64)
        self.bn31   = nn.BatchNorm1d(32)
        self.drop   = nn.Dropout(p=dropout)
        self.fc32   = nn.Linear(32, num_classes)

        self.apply(kaiming_init)

    def extract_feat(self, x):
        x = self.mfm(self.conv1(x))       # 16 -> 8
        x = self.pool3(x)

        x = self.mfm(self.conv4(x))       # 16 -> 8
        x = self.bn6(x)
        x = self.mfm(self.conv7(x))       # 24 -> 12
        x = self.pool9(x)
        x = self.bn10(x)

        x = self.mfm(self.conv11(x))      # 24 -> 12
        x = self.bn13(x)
        x = self.mfm(self.conv14(x))      # 24 -> 12
        x = self.pool16(x)

        x = self.mfm(self.conv17(x))      # 24 -> 12
        x = self.bn19(x)
        x = self.mfm(self.conv20(x))      # 8 -> 4
        x = self.bn22(x)
        x = self.mfm(self.conv23(x))      # 8 -> 4
        x = self.bn25(x)
        x = self.mfm(self.conv26(x))      # 8 -> 4
        x = self.pool28(x)

        x = torch.flatten(x, 1)
        x = self.fc29(x)                  # (B,64)

        a, b = torch.split(x, 32, dim=1)
        x = torch.max(a, b)           # (B,32)

        x = self.bn31(x)
        x = self.drop(x)
        return x  # embedding (B,32)

    def forward(self, x):
        return self.fc32(self.extract_feat(x))

In [30]:
class SEBasicBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, reduction=16):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.se    = SEBlock(out_ch, reduction=reduction)

        self.downsample = None
        if stride != 1 or in_ch != out_ch:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch)
            )

    def forward(self, x):
        identity = x
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.bn2(self.conv2(out))
        out = self.se(out)

        if self.downsample is not None:
            identity = self.downsample(identity)

        out = F.relu(out + identity, inplace=True)
        return out

class SENet12(nn.Module):
    """
    SENet12 from table:
    Conv7x7 s2 -> BN -> ReLU -> MaxPool3x3 s2
    SEResNet Module x1 (16)
    SEResNet Module x2 (32, first stride=2)
    SEResNet Module x3 (64, first stride=2)
    SEResNet Module x1 (128, first stride=2)
    GlobalAvgPool -> FC(2)
    """
    def __init__(self, num_classes=2, reduction=16):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
        )

        self.layer1 = self._make_layer(16, 16, blocks=1, stride=1, reduction=reduction)
        self.layer2 = self._make_layer(16, 32, blocks=2, stride=2, reduction=reduction)
        self.layer3 = self._make_layer(32, 64, blocks=3, stride=2, reduction=reduction)
        self.layer4 = self._make_layer(64, 128, blocks=1, stride=2, reduction=reduction)

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc   = nn.Linear(128, num_classes)

        self.apply(kaiming_init)

    def _make_layer(self, in_ch, out_ch, blocks, stride, reduction):
        layers = [SEBasicBlock(in_ch, out_ch, stride=stride, reduction=reduction)]
        for _ in range(1, blocks):
            layers.append(SEBasicBlock(out_ch, out_ch, stride=1, reduction=reduction))
        return nn.Sequential(*layers)

    def extract_feat(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x).flatten(1)
        return x

    def forward(self, x):
        x = self.extract_feat(x)
        x = self.fc(x)
        return x

In [31]:
import math
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

# -------------------------
# A-Softmax (SphereFace) head
# -------------------------
class ASoftmaxLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int, m: int = 4):
        super().__init__()
        assert m in [1, 2, 3, 4, 5], "Implementing m up to 5 is typical; m=4 recommended here."
        self.in_features = in_features
        self.out_features = out_features
        self.m = m
        self.W = nn.Parameter(torch.randn(out_features, in_features))
        nn.init.xavier_normal_(self.W)

    def _cos_m_theta(self, cos_theta: torch.Tensor):
        # cos(mθ) using multiple-angle formulas (Chebyshev polynomials)
        m = self.m
        if m == 1:
            return cos_theta
        elif m == 2:
            return 2*cos_theta**2 - 1
        elif m == 3:
            return 4*cos_theta**3 - 3*cos_theta
        elif m == 4:
            return 8*cos_theta**4 - 8*cos_theta**2 + 1
        elif m == 5:
            return 16*cos_theta**5 - 20*cos_theta**3 + 5*cos_theta
        else:
            raise ValueError("Unsupported m")

    def forward(self, x: torch.Tensor, y: torch.Tensor):
        """
        x: (B, in_features)
        y: (B,)
        returns: logits (B, out_features)
        """
        # normalize
        x_norm = torch.norm(x, p=2, dim=1, keepdim=True).clamp_min(1e-8)  # (B,1)
        x_hat = x / x_norm                                                # (B,F)

        W_hat = F.normalize(self.W, p=2, dim=1)                            # (C,F)

        cos_theta = torch.matmul(x_hat, W_hat.t()).clamp(-1.0, 1.0)        # (B,C)
        cos_m_theta = self._cos_m_theta(cos_theta)                         # (B,C)

        # SphereFace "phi(theta)" correction for monotonicity:
        # phi = (-1)^k * cos(mθ) - 2k, where k = floor(m*θ/pi)
        theta = torch.acos(cos_theta)                                      # (B,C)
        k = torch.floor(self.m * theta / math.pi)                          # (B,C)
        phi_theta = ((-1.0)**k) * cos_m_theta - 2.0*k

        # scale by ||x||
        logits = cos_theta * x_norm                                         # (B,C)
        # replace target logits with phi * ||x||
        idx = torch.arange(x.size(0), device=x.device)
        logits[idx, y] = (phi_theta[idx, y] * x_norm[idx, 0])

        return logits


@torch.no_grad()
def bonafide_cosine_score(features: torch.Tensor, W: torch.Tensor, bonafide_class: int = 0):
    """
    features: (B, F)
    W: (num_classes, F) unnormalized weights from last layer/head
    score = cosine(feature, W_bonafide)
    """
    f = F.normalize(features, p=2, dim=1)
    w = F.normalize(W[bonafide_class:bonafide_class+1], p=2, dim=1)  # (1,F)
    return (f * w).sum(dim=1)  # (B,)

class LCNNForASoftmax(nn.Module):
    def __init__(self, base_lcnn: nn.Module, embed_dim: int, num_classes: int = 2, m: int = 4):
        super().__init__()
        self.backbone = base_lcnn
        self.head = ASoftmaxLinear(embed_dim, num_classes, m=m)

    def forward(self, x, y=None, return_feat=False):
        feat = self.backbone.extract_feat(x)  # (B, embed_dim)
        if return_feat:
            return feat
        assert y is not None, "y is required for A-Softmax forward"
        logits = self.head(feat, y)
        return logits

In [32]:
def load_optionA_lcnn_big(ckpt_path, device):
    base = LCNNBig(dropout=0.75)
    model = LCNNForASoftmax(base, embed_dim=80, num_classes=2, m=4)
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model"], strict=True)
    return model

def load_optionA_lcnn_small(ckpt_path, device):
    base = LCNNSmall(dropout=0.75)
    model = LCNNForASoftmax(base, embed_dim=32, num_classes=2, m=4)
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model"], strict=True)
    return model

def load_optionA_senet12(ckpt_path, device):
    model = SENet12(num_classes=2, reduction=16)
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model"], strict=True)
    return model

In [33]:
def compute_eer(labels, scores):
    labels = np.asarray(labels).astype(int)
    scores = np.asarray(scores).astype(float)

    idx = np.argsort(scores)[::-1]
    labels = labels[idx]
    scores = scores[idx]

    pos = (labels == 0).astype(int)  # genuine
    neg = (labels == 1).astype(int)  # fake
    P, N = pos.sum(), neg.sum()
    if P == 0 or N == 0:
        return 1.0, 0.0

    tp = np.cumsum(pos)
    fp = np.cumsum(neg)
    fn = P - tp
    tn = N - fp

    far = fp / (fp + tn + 1e-12)   # fake accepted as genuine
    frr = fn / (fn + tp + 1e-12)   # genuine rejected

    i = np.argmin(np.abs(far - frr))
    eer = 0.5 * (far[i] + frr[i])
    thr = scores[i]
    return float(eer), float(thr)

def weer(eer_r1, eer_r2, alpha=0.4, beta=0.6):
    return alpha * eer_r1 + beta * eer_r2

@torch.no_grad()
def bonafide_cosine_score(features: torch.Tensor, W: torch.Tensor, bonafide_class: int = 0):
    f = F.normalize(features, p=2, dim=1)
    w = F.normalize(W[bonafide_class:bonafide_class+1], p=2, dim=1)  # (1,F)
    return (f * w).sum(dim=1)  # (B,)

In [34]:
@torch.no_grad()
def evaluate_model(
    encoder,
    classifier_name,
    classifier_ckpt,
    r1_dir, r2_dir,
    label_R1_df, label_R2_df,
    device="cuda",
    crop_sec=2.0,
    sr=16000,
    num_crops=10,
    seed=0,
    batch_size=32,
    num_workers=0,
    image_hw=(16,16),
    alpha=0.4,
    beta=0.6,
):
    # reproducibility
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    crop_len = int(sr * crop_sec)

    ds1 = RandomCropTestDatasetNoNorm(r1_dir, label_R1_df, crop_len=crop_len)
    ds2 = RandomCropTestDatasetNoNorm(r2_dir, label_R2_df, crop_len=crop_len)

    # classifier load
    if classifier_name == "lcnn_big":
        clf = load_optionA_lcnn_big(classifier_ckpt, device)
    elif classifier_name == "lcnn_small":
        clf = load_optionA_lcnn_small(classifier_ckpt, device)
    elif classifier_name == "senet12":
        clf = load_optionA_senet12(classifier_ckpt, device)
    else:
        raise ValueError(f"Unknown classifier_name: {classifier_name}")

    eer1, thr1 = eval_round_optionA(
        encoder=encoder,
        classifier_name=classifier_name,
        classifier_model=clf,
        base_ds=ds1,
        num_crops=num_crops,
        device=device,
        batch_size=batch_size,
        num_workers=num_workers,
        image_hw=image_hw,
    )

    eer2, thr2 = eval_round_optionA(
        encoder=encoder,
        classifier_name=classifier_name,
        classifier_model=clf,
        base_ds=ds2,
        num_crops=num_crops,
        device=device,
        batch_size=batch_size,
        num_workers=num_workers,
        image_hw=image_hw,
    )

    result = {
        "model": classifier_name,
        "ckpt": classifier_ckpt,
        "EER_R1": eer1,
        "THR_R1": thr1,
        "EER_R2": eer2,
        "THR_R2": thr2,
        "WEER": alpha * eer1 + beta * eer2,
    }

    print(f"\n=== Test Result: {classifier_name} ===")
    print(f"ckpt    : {classifier_ckpt}")
    print(f"EER_R1  : {eer1:.5f}")
    print(f"EER_R2  : {eer2:.5f}")
    print(f"WEER    : {result['WEER']:.5f}")

    return result

In [35]:
def load_pretrained_encoder(ckpt_path, device):
  ckpt = torch.load(ckpt_path, map_location=device)

  cfg = ckpt["config"]
  encoder = Encoder(
    channel_list=cfg["channel_list"],
    kernel_list=cfg["kernel_list"],
    stride_list=cfg["stride_list"],
    embedding_dim=embedding_dim,
    worker_input_dim=worker_input_dim,
    proj_hidden=proj_hidden,
    tcn_layers=cfg["tcn_layers"],
  ).to(device)

  encoder.load_state_dict(ckpt["encoder"], strict=False)
  encoder.eval()

  return encoder

In [ ]:
import os
import pandas as pd

def record_test_result(result, save_csv="checkpoints/stage2_optionA/test_results.csv"):
    os.makedirs(os.path.dirname(save_csv), exist_ok=True)

    row_df = pd.DataFrame([result])

    if os.path.exists(save_csv):
        prev = pd.read_csv(save_csv)
        out = pd.concat([prev, row_df], ignore_index=True)
    else:
        out = row_df

    out.to_csv(save_csv, index=False)
    print(f"[INFO] saved test result to: {save_csv}")

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ENCODER_CKPT = "checkpoints/pretrain_encoder_final_epoch100.pt"
encoder = load_pretrained_encoder(ENCODER_CKPT, DEVICE)

R1_DIR = "data/test/R1/librosa"
R2_DIR = "data/test/R2/librosa"

model_ckpts = {
    "lcnn_big":   "checkpoints/stage2_optionA/lcnn_big_best.pt",
    "lcnn_small": "checkpoints/stage2_optionA/lcnn_small_best.pt",
    "senet12":    "checkpoints/stage2_optionA/senet12_best.pt",
}

all_results = {}

for model_name, ckpt_path in model_ckpts.items():
    result = evaluate_single_optionA_model(
        encoder=encoder,
        classifier_name=model_name,
        classifier_ckpt=ckpt_path,
        r1_dir=R1_DIR,
        r2_dir=R2_DIR,
        label_R1_df=label_R1_df,
        label_R2_df=label_R2_df,
        device=DEVICE,
        num_crops=10,
        seed=0,
        image_hw=(16,16),
    )
    record_test_result(result)
    all_results[model_name] = result

Eval lcnn_small:  60%|█████▉    | 2845/4764 [06:19<04:02,  7.91it/s]